# Detector Bake-off Spike — RF-DETR vs YOLOX vs R2-cameron

**Goal:** answer one question cheaply — *does a permissively-licensed detector clear our detection bar on our own data?* — before committing to an AGPL escape / Stage-1 port.

Three detectors are scored **through the same engine-agnostic evaluator** (a verbatim port of `src/utils/det_match.match_predictions`, greedy best-first IoU matching at `iou=0.50`) on the **same val + test tiles**:

| Row | Licence | Notes |
|---|---|---|
| **R2-cameron (whole-tile)** | AGPL | baseline, **re-measured live in this notebook** — we don't trust the registry's recorded metrics. Whole-tile = fair apples-to-apples vs the others. |
| **RF-DETR** | Apache-2.0 | the bet — built for fast fine-tuning on small custom datasets, aligns with your Roboflow workflow. |
| **YOLOX** | Apache-2.0 | simple fallback. *Setup is version-sensitive — see caveat on that cell.* |

**What clearing looks like:** RF-DETR's AP@0.50 / best-F1 lands at or above R2-cameron's — where both are measured *here, on the same tiles, by the same evaluator*. Everything is self-contained, so no external/registry number is relied on. If RF-DETR clears the freshly-measured R2 baseline → greenlight the port, drop the $5k licence. If it craters → you learned it in a couple of Colab-Pro runs.

---
### Before you run — nothing to upload
This reuses the **same Drive as `colab_ramp_training.ipynb`**: `/content/drive/MyDrive/solar-soiling`. It reads the `naip.zip` and `models/r2_cameron_20260509.pt` that are already there, extracting the dataset with the same `valid`->`val` restructure your ramp cell uses. Just set Runtime → **GPU** and Run all. (If your Drive folder differs, edit `DRIVE` in the Config cell.)

In [ ]:
# --- Runtime + Drive + config (reuses your ramp-test Drive) --------------
import os, shutil, zipfile, torch
from pathlib import Path
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "!! CPU-only — set Runtime>GPU")

from google.colab import drive
drive.mount('/content/drive')

DRIVE            = '/content/drive/MyDrive/solar-soiling'          # same as colab_ramp_training.ipynb
NAIP_ZIP         = f'{DRIVE}/naip.zip'                             # already staged for the ramp runs
R2_WEIGHTS_DRIVE = f'{DRIVE}/models/r2_cameron_20260509.pt'
WORK             = '/content/bakeoff'
os.makedirs(WORK, exist_ok=True)

# Eval + train knobs ------------------------------------------------------
IOU_THRESH  = 0.50     # your production iou
CONF_REPORT = 0.40     # your production operating point (F1 also swept for a fair cross-model number)
EPOCHS      = 60       # per run; a few minutes at this dataset size on Colab Pro
IMG_SIZE    = 640

assert os.path.isfile(NAIP_ZIP), f"naip.zip not found at {NAIP_ZIP}"
assert os.path.isfile(R2_WEIGHTS_DRIVE), f"weights not found at {R2_WEIGHTS_DRIVE}"
R2_WEIGHTS = f'{WORK}/r2_cameron_20260509.pt'
shutil.copy(R2_WEIGHTS_DRIVE, R2_WEIGHTS)
print("Weights + naip.zip found on Drive — nothing to re-upload.")

In [ ]:
# --- Extract naip.zip -> local YOLO layout (same restructure as ramp cell 4)
NAIP_DIR = f'{WORK}/naip'
shutil.rmtree(NAIP_DIR, ignore_errors=True)
tmp = Path(f'{WORK}/naip_extract'); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
print("Extracting naip.zip ...")
with zipfile.ZipFile(NAIP_ZIP) as z:
    z.extractall(str(tmp))

split_map = {'train': 'train', 'valid': 'val', 'test': 'test'}   # Roboflow 'valid' -> 'val'
found = {}
for root, dirs, _ in os.walk(str(tmp)):
    for d in dirs:
        if d in split_map and d not in found:
            found[d] = Path(root)/d
    if len(found) == 3:
        break
base = Path(NAIP_DIR)
for src_name, dst_name in split_map.items():
    src = found.get(src_name)
    if not src:
        continue
    for sub in ('images', 'labels'):
        s = src/sub; dst = base/sub/dst_name
        if s.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(str(s), str(dst))
shutil.rmtree(tmp, ignore_errors=True)

NAIP = Path(NAIP_DIR)
SPLITS = {s: (NAIP/'images'/s, NAIP/'labels'/s) for s in ['train', 'val', 'test']}
for s, (i, l) in SPLITS.items():
    n_img = len(list(i.glob('*.png')) + list(i.glob('*.jpg'))) if i.exists() else 0
    n_lbl = len(list(l.glob('*.txt'))) if l.exists() else 0
    print(f"{s:5s}  images={n_img:4d}  labels={n_lbl:4d}")
print("Expected (from your ramp notebook): train=171 val=27 test=50")

In [ ]:
# --- Engine-agnostic evaluator (verbatim port of src/utils/det_match) ----
import numpy as np
from PIL import Image

def _iou(a, b):  # identical semantics to src/utils/det_match.iou
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = max(0.0, ax2-ax1)*max(0.0, ay2-ay1) + max(0.0, bx2-bx1)*max(0.0, by2-by1) - inter
    return inter/union if union > 0 else 0.0

def match_predictions(preds, gts, iou_thresh=0.5):
    # verbatim port: greedy, preds assumed conf-desc, each GT claimed once
    matched, tps, fps = set(), [], []
    for pi, pb in enumerate(preds):
        best_iou, best_gt = 0.0, None
        for gi, gb in enumerate(gts):
            if gi in matched:
                continue
            v = _iou(pb, gb)
            if v > best_iou:
                best_iou, best_gt = v, gi
        if best_gt is not None and best_iou >= iou_thresh:
            tps.append((pi, best_gt, best_iou)); matched.add(best_gt)
        else:
            fps.append(pi)
    fns = [i for i in range(len(gts)) if i not in matched]
    return tps, fps, fns

def load_gt_norm(label_path):
    # YOLO-seg .txt -> [(x1,y1,x2,y2) normalized]; mirrors rca.parse_label_polys_norm + poly bbox
    if not Path(label_path).exists():
        return []
    out = []
    for line in Path(label_path).read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        try:
            c = np.array([float(p) for p in parts[1:]], dtype=np.float64)
        except ValueError:
            continue
        xs, ys = c[0::2], c[1::2]
        if len(xs) < 3:
            continue
        out.append((float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())))
    return out

def image_size(p):
    with Image.open(p) as im:
        return im.size  # (w, h)

def _f1(tp, fp, fn):
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f

def evaluate(preds_by_img, gt_by_img):
    # preds_by_img: {stem: [(score,(x1,y1,x2,y2)_norm),...]}  gt_by_img: {stem: [box_norm,...]}
    # --- AP@0.50 (COCO-style, 101-pt), threshold-free -> the fair cross-model number
    scored, n_gt = [], 0
    for stem, gts in gt_by_img.items():
        n_gt += len(gts)
        dets = sorted(preds_by_img.get(stem, []), key=lambda d: -d[0])
        _tp, _fp, _ = match_predictions([b for _, b in dets], gts, IOU_THRESH)
        tp_idx = {pi for pi, _, _ in _tp}
        for i, (sc, _) in enumerate(dets):
            scored.append((sc, 1 if i in tp_idx else 0))
    scored.sort(key=lambda x: -x[0])
    tp_c = fp_c = 0; rec, prec = [], []
    for _, is_tp in scored:
        tp_c += is_tp; fp_c += (1-is_tp)
        rec.append(tp_c/n_gt if n_gt else 0.0)
        prec.append(tp_c/(tp_c+fp_c))
    ap = 0.0
    for t in np.linspace(0, 1, 101):
        ps = [p for p, r in zip(prec, rec) if r >= t]
        ap += (max(ps) if ps else 0.0)/101
    # --- best-F1 over a conf sweep (fair across differing score calibrations)
    best = (0.0, 0.0, 0.0, 0.0)  # f1, conf, p, r
    at_report = (0.0, 0.0, 0.0)
    for conf in np.linspace(0.05, 0.95, 19):
        TP = FP = FN = 0
        for stem, gts in gt_by_img.items():
            dets = sorted([d for d in preds_by_img.get(stem, []) if d[0] >= conf], key=lambda d: -d[0])
            tp, fp, fn = match_predictions([b for _, b in dets], gts, IOU_THRESH)
            TP += len(tp); FP += len(fp); FN += len(fn)
        p, r, f = _f1(TP, FP, FN)
        if f > best[0]:
            best = (f, float(conf), p, r)
        if abs(conf - CONF_REPORT) < 0.026:
            at_report = (p, r, f)
    return {"ap50": round(ap, 3), "best_f1": round(best[0], 3), "best_conf": round(best[1], 2),
            "f1_at_report": round(at_report[2], 3), "p_at_report": round(at_report[0], 3),
            "r_at_report": round(at_report[1], 3), "n_gt": n_gt}

def split_paths(split):
    i, _ = SPLITS[split]
    return sorted(list(i.glob('*.png')) + list(i.glob('*.jpg')))

def gt_for(split):
    _, l = SPLITS[split]
    return {p.stem: load_gt_norm(l/(p.stem+'.txt')) for p in split_paths(split)}

def run_eval(predict_fn, splits=('val', 'test')):
    rows = {}
    for s in splits:
        paths = split_paths(s)
        rows[s] = evaluate(predict_fn(paths), gt_for(s))
        print(f"  [{s}] {rows[s]}")
    return rows

print("evaluator ready")

In [ ]:
# --- Baseline: R2-cameron, whole-tile (no SAHI) --------------------------
# AGPL, but this is eval-only baseline measurement, not shipped product.
!pip -q install ultralytics
from ultralytics import YOLO
_r2 = YOLO(R2_WEIGHTS, task='segment')

def predict_r2(img_paths):
    out = {}
    for p in img_paths:
        r = _r2.predict(source=str(p), imgsz=IMG_SIZE, conf=0.01, iou=0.7, verbose=False)[0]
        w, h = image_size(p)
        dets = []
        if r.boxes is not None and len(r.boxes) > 0:
            xyxy = r.boxes.xyxy.cpu().numpy(); conf = r.boxes.conf.cpu().numpy()
            for (x1, y1, x2, y2), c in zip(xyxy, conf):
                dets.append((float(c), (x1/w, y1/h, x2/w, y2/h)))
        out[p.stem] = dets
    return out

print("R2-cameron (whole-tile, no SAHI) -- measured fresh here, this is THE baseline:")
R2 = run_eval(predict_r2)

## RF-DETR — the bet
Apache-2.0, built for fast fine-tuning on small custom datasets. Consumes COCO format (the same format Roboflow exports), so the converter below doubles as your Roboflow-alignment glue.

**W1 fair rematch (`docs/PERMISSIVE_STACK_MIGRATION.md` §W1):** the 1st cold run (res 560, 60ep) lost to R2 — but unfairly, since R2 rode a domain warm-start chain (sahi_baseline → R0 → R2) and RF-DETR got one cold low-res pass. This run levels it: **resolution 728**, **batch 4 / grad-accum 4** (effective batch 16 within Colab VRAM), **~100 epochs**, with a **convergence check** on the per-epoch `valid` AP so we don't compare an undertrained model. Eval scores the **best** checkpoint. Read the result only after the convergence cell says *plateaued*, then judge it against **Gate A** in the Results section.

In [ ]:
# --- YOLO-seg -> COCO converter (shared by RF-DETR + YOLOX) --------------
import json, shutil

def yolo_to_coco(split, dst_dir, copy_images=True):
    img_dir, lbl_dir = SPLITS[split]
    os.makedirs(dst_dir, exist_ok=True)
    images, annos, ann_id = [], [], 1
    paths = sorted(list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg')))
    for iid, p in enumerate(paths, 1):
        w, h = image_size(p)
        images.append({"id": iid, "file_name": p.name, "width": w, "height": h})
        if copy_images:
            shutil.copy(p, Path(dst_dir)/p.name)
        for (x1, y1, x2, y2) in load_gt_norm(lbl_dir/(p.stem+'.txt')):  # normalized bbox
            bx, by, bw, bh = x1*w, y1*h, (x2-x1)*w, (y2-y1)*h
            annos.append({"id": ann_id, "image_id": iid, "category_id": 1,
                          "bbox": [bx, by, bw, bh], "area": bw*bh, "iscrowd": 0})
            ann_id += 1
    coco = {"images": images, "annotations": annos,
            "categories": [{"id": 1, "name": "solar_array", "supercategory": "none"}]}
    with open(Path(dst_dir)/'_annotations.coco.json', 'w') as f:
        json.dump(coco, f)
    return len(images), len(annos)

# RF-DETR / Roboflow layout: dataset/{train,valid,test}/(images + _annotations.coco.json)
RFDIR = f'{WORK}/rfdetr_data'
for split, dst in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
    n_i, n_a = yolo_to_coco(split, f'{RFDIR}/{dst}')
    print(f"{dst:6s} images={n_i:4d} boxes={n_a:5d}")

In [ ]:
# --- RF-DETR train (W1 fair rematch) -------------------------------------
# NB: training deps (pytorch_lightning etc.) live in the [train] extra — plain
# `pip install rfdetr` only pulls inference deps and .train() will ModuleNotFoundError.
!pip -q install "rfdetr[train,loggers]"
from rfdetr import RFDETRBase

# W1 fair-rematch knobs (see docs/PERMISSIVE_STACK_MIGRATION.md §W1). The 1st cold
# run was res 560 / 60ep and lost to R2 — but R2 rode a domain warm-start chain, so
# that comparison was not fair. This gives RF-DETR a real budget before Gate A:
#   - resolution 728  : higher input res for small-panel recall (must be a multiple
#                       of 56; default 560). Passed to the CONSTRUCTOR, not .train().
#   - batch_size=4, grad_accum_steps=4 : fit 728px in Colab VRAM while holding the
#                       effective batch at 16 (=4x4), close to the 1st run's 8x2=16.
#   - RFDETR_EPOCHS ~100 : real budget; watch for overfit at 171 train imgs — the
#                       convergence-check cell below decides if this was enough.
RFDETR_RES    = 728          # multiple of 56
RFDETR_EPOCHS = 100
RFDETR_OUT    = f'{WORK}/rfdetr_out'
assert RFDETR_RES % 56 == 0, "RF-DETR resolution must be a multiple of 56"

rf = RFDETRBase(resolution=RFDETR_RES)
rf.train(dataset_dir=RFDIR, epochs=RFDETR_EPOCHS, batch_size=4,
         grad_accum_steps=4, lr=1e-4, output_dir=RFDETR_OUT)

In [ ]:
# --- RF-DETR eval ---------------------------------------------------------
# Fair rematch: score the BEST checkpoint (peak val AP from the convergence cell),
# not necessarily the last epoch. RF-DETR writes checkpoint_best_*.pth into RFDETR_OUT.
# Fall back to the in-memory (last-epoch) weights if no best checkpoint is found.
_best = next((Path(RFDETR_OUT)/c for c in
              ('checkpoint_best_ema.pth', 'checkpoint_best_total.pth', 'checkpoint_best_regular.pth')
              if (Path(RFDETR_OUT)/c).exists()), None)
if _best is not None:
    print(f"eval: loading best checkpoint {_best.name}")
    try:
        rf = RFDETRBase(pretrain_weights=str(_best), resolution=RFDETR_RES)
    except Exception as e:
        print(f"  (could not reload best ckpt: {e} — using in-memory last-epoch weights)")
else:
    print("eval: no best checkpoint found — using in-memory last-epoch weights")

def predict_rfdetr(img_paths):
    out = {}
    for p in img_paths:
        img = Image.open(p).convert('RGB')
        det = rf.predict(img, threshold=0.01)   # supervision.Detections: xyxy px, confidence, class_id
        w, h = img.size
        out[p.stem] = [(float(c), (x1/w, y1/h, x2/w, y2/h))
                       for (x1, y1, x2, y2), c in zip(det.xyxy, det.confidence)]
    return out

print("RF-DETR:")
RF = run_eval(predict_rfdetr)

In [ ]:
# --- RF-DETR eval ---------------------------------------------------------
def predict_rfdetr(img_paths):
    out = {}
    for p in img_paths:
        img = Image.open(p).convert('RGB')
        det = rf.predict(img, threshold=0.01)   # supervision.Detections: xyxy px, confidence, class_id
        w, h = img.size
        out[p.stem] = [(float(c), (x1/w, y1/h, x2/w, y2/h))
                       for (x1, y1, x2, y2), c in zip(det.xyxy, det.confidence)]
    return out

print("RF-DETR:")
RF = run_eval(predict_rfdetr)

## YOLOX — simple fallback

> **Caveat — read before running.** YOLOX's install/train path is version-sensitive (Megvii repo, custom `Exp` config, occasional torch/numpy pin conflicts on Colab) and is **not** as clean as RF-DETR's `.train()`. This cell is best-effort scaffolding; if RF-DETR already clears the bar you can skip YOLOX entirely. If a step errors, it's almost always a dependency pin — treat this as a starting point, not a guaranteed one-shot.

In [ ]:
# --- YOLOX: COCO layout (standard single-json + image dirs) --------------
YXDIR = f'{WORK}/yolox_data'
os.makedirs(f'{YXDIR}/annotations', exist_ok=True)
for split, name in [('train', 'train2017'), ('val', 'val2017')]:
    d = f'{YXDIR}/{name}'
    n_i, n_a = yolo_to_coco(split, d)
    shutil.move(f'{d}/_annotations.coco.json', f'{YXDIR}/annotations/instances_{name}.json')
    print(f"{name}: images={n_i} boxes={n_a}")

In [ ]:
# --- YOLOX: install + generate Exp + train -------------------------------
%cd {WORK}
![ -d YOLOX ] || git clone -q https://github.com/Megvii-BaseDetection/YOLOX.git
%cd YOLOX
# YOLOX setup.py pins onnx-simplifier==0.4.10, which fails to build on py3.12 and aborts the
# whole install (leaving loguru/thop uninstalled). ONNX is export-only, so install core with
# --no-deps + just the runtime deps YOLOX actually imports.
!pip -q install -e . --no-deps
!pip -q install loguru thop ninja tabulate
!wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth -O yolox_s.pth

exp_lines = [
    "import os",
    "from yolox.exp import Exp as MyBaseExp",
    "class Exp(MyBaseExp):",
    "    def __init__(self):",
    "        super().__init__()",
    "        self.num_classes = 1",
    "        self.depth = 0.33",
    "        self.width = 0.50",
    f"        self.data_dir = '{YXDIR}'",
    "        self.train_ann = 'instances_train2017.json'",
    "        self.val_ann = 'instances_val2017.json'",
    f"        self.max_epoch = {EPOCHS}",
    f"        self.input_size = ({IMG_SIZE}, {IMG_SIZE})",
    f"        self.test_size = ({IMG_SIZE}, {IMG_SIZE})",
    "        self.mosaic_prob = 0.5",
    "        self.enable_mixup = False",
    "        self.exp_name = 'yolox_s_solar'",
]
open(f'{WORK}/yolox_solar_exp.py', 'w').write("\n".join(exp_lines))
!python tools/train.py -f {WORK}/yolox_solar_exp.py -d 1 -b 8 --fp16 -c yolox_s.pth

In [ ]:
# --- YOLOX eval -----------------------------------------------------------
# Loads the trained checkpoint and scores it through the same evaluator.
import torch
from yolox.exp import get_exp
from yolox.utils import postprocess
from yolox.data.data_augment import ValTransform

_exp = get_exp(f'{WORK}/yolox_solar_exp.py', None)
_net = _exp.get_model().cuda().eval()
_ck = torch.load(f'{WORK}/YOLOX/YOLOX_outputs/yolox_s_solar/best_ckpt.pth', map_location='cpu')
_net.load_state_dict(_ck['model'])
_pre = ValTransform(legacy=False)

def predict_yolox(img_paths):
    out = {}
    for p in img_paths:
        import cv2
        im = cv2.imread(str(p)); h, w = im.shape[:2]
        ratio = min(IMG_SIZE/h, IMG_SIZE/w)
        t, _ = _pre(im, None, (IMG_SIZE, IMG_SIZE))
        t = torch.from_numpy(t).unsqueeze(0).float().cuda()
        with torch.no_grad():
            o = postprocess(_net(t), 1, conf_thre=0.01, nms_thre=0.65)[0]
        dets = []
        if o is not None:
            o = o.cpu().numpy()
            for r in o:
                x1, y1, x2, y2 = r[0:4]/ratio
                sc = float(r[4]*r[5])
                dets.append((sc, (x1/w, y1/h, x2/w, y2/h)))
        out[p.stem] = dets
    return out

print("YOLOX:")
YX = run_eval(predict_yolox)

# --- Comparison table -----------------------------------------------------
import pandas as pd
def rows_for(name, res):
    return [{"model": name, "split": s,
             "AP@0.50": r["ap50"], "best_F1": r["best_f1"], "@conf": r["best_conf"],
             f"F1@{CONF_REPORT}": r["f1_at_report"],
             "P": r["p_at_report"], "R": r["r_at_report"], "n_gt": r["n_gt"]}
            for s, r in res.items()]

data = []
for name, var in [("R2-cameron (whole-tile)", "R2"), ("RF-DETR", "RF"), ("YOLOX", "YX")]:
    if var in globals():
        data += rows_for(name, globals()[var])

df = pd.DataFrame(data)
print(df.to_string(index=False))
print("\nAll three measured here on the same tiles by the same evaluator -- compare rows directly.")

# --- Gate A verdict (docs/PERMISSIVE_STACK_MIGRATION.md §4) ---------------
# Question: does a fairly-trained permissive detector reach R2-cameron's freshly-measured
# quality on the TEST split? Abs bar: AP@50 >= 0.60 and best-F1 >= 0.63. Also compare to
# R2's own freshly-measured test numbers (the real parity target), within noise.
GATE_AP, GATE_F1 = 0.60, 0.63
print("\n" + "=" * 64 + "\nGATE A (test split) — permissive-detector parity check\n" + "=" * 64)
if 'RF' in globals() and 'test' in RF:
    rf_ap, rf_f1 = RF['test']['ap50'], RF['test']['best_f1']
    r2_ap = R2['test']['ap50'] if 'R2' in globals() and 'test' in R2 else None
    r2_f1 = R2['test']['best_f1'] if 'R2' in globals() and 'test' in R2 else None
    print(f"RF-DETR test : AP@50 {rf_ap:.3f}  best-F1 {rf_f1:.3f}")
    if r2_ap is not None:
        print(f"R2 test      : AP@50 {r2_ap:.3f}  best-F1 {r2_f1:.3f}   (fresh parity target)")
    hits_abs = (rf_ap >= GATE_AP) and (rf_f1 >= GATE_F1)
    near_r2  = (r2_ap is not None) and (rf_ap >= r2_ap - 0.03) and (rf_f1 >= r2_f1 - 0.03)
    if hits_abs or near_r2:
        verdict = "PASS — reaches parity (within noise). Proceed W2->W6; port justified."
    elif r2_ap is not None and (rf_ap >= r2_ap - 0.10) and (rf_f1 >= r2_f1 - 0.10):
        verdict = "PARTIAL — closes most of the gap. Continue to W2/W3 (30cm data), then re-gate."
    else:
        verdict = "FAIL — still far behind after a fair run. Escalate to Cameron (§4)."
    print(f"abs bar (AP>=0.60 & F1>=0.63): {'yes' if hits_abs else 'no'}   within-noise of R2: {'yes' if near_r2 else 'no'}")
    print(f"VERDICT: {verdict}")
else:
    print("RF-DETR test results not present — run the RF-DETR train+eval cells first.")

In [ ]:
# --- Comparison table -----------------------------------------------------
import pandas as pd
def rows_for(name, res):
    return [{"model": name, "split": s,
             "AP@0.50": r["ap50"], "best_F1": r["best_f1"], "@conf": r["best_conf"],
             f"F1@{CONF_REPORT}": r["f1_at_report"],
             "P": r["p_at_report"], "R": r["r_at_report"], "n_gt": r["n_gt"]}
            for s, r in res.items()]

data = []
for name, var in [("R2-cameron (whole-tile)", "R2"), ("RF-DETR", "RF"), ("YOLOX", "YX")]:
    if var in globals():
        data += rows_for(name, globals()[var])

df = pd.DataFrame(data)
print(df.to_string(index=False))
print("\nAll three measured here on the same tiles by the same evaluator -- compare rows directly.")
print("Read: if RF-DETR's AP@0.50 / best_F1 >= R2-cameron's (test split), a permissive detector holds on your data => port is de-risked.")

## Area half — SAM mask quality (box -> mask -> minAreaRect -> m2)

Validates the *second* stage of the proposed stack. We prompt SAM with the **ground-truth boxes** (not a detector's) on purpose — that isolates SAM's mask quality from any detector error, i.e. it measures the *ceiling* of the area pipeline. Each SAM mask is compared to your real GT polygon (IoU + area error), and we also compute the production recipe `minAreaRect -> m2`.

> Uses **SAM2** (`facebook/sam2-hiera-large`, HF auto-download). **SAM3 drops into the exact same `set_image` / `predict(box=...)` role** if/when you confirm it's released + permissive — swap the two lines, nothing else changes.

In [ ]:
# --- SAM area-quality validation -----------------------------------------
!pip -q install sam2
import cv2
from sam2.sam2_image_predictor import SAM2ImagePredictor
predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2-hiera-large")  # <- SAM3 goes here

GSD_M      = 0.6      # NAIP ~0.6 m/px; pull per-tile from tile_index.json for real m2
N_SAMPLE   = 60       # cap arrays for speed
EVAL_SPLIT = 'val'

def load_gt_polys_norm(label_path):
    polys = []
    if not Path(label_path).exists():
        return polys
    for line in Path(label_path).read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        c = np.array([float(p) for p in parts[1:]], dtype=np.float64)
        xs, ys = c[0::2], c[1::2]
        if len(xs) >= 3:
            polys.append(np.column_stack([xs, ys]))
    return polys

records, examples = [], []
_, lbl_dir = SPLITS[EVAL_SPLIT]
for p in split_paths(EVAL_SPLIT):
    if len(records) >= N_SAMPLE:
        break
    polys = load_gt_polys_norm(lbl_dir/(p.stem+'.txt'))
    if not polys:
        continue
    im = np.array(Image.open(p).convert('RGB')); H, W = im.shape[:2]
    predictor.set_image(im)
    for poly in polys:
        if len(records) >= N_SAMPLE:
            break
        pk = poly * np.array([W, H])
        box = np.array([pk[:,0].min(), pk[:,1].min(), pk[:,0].max(), pk[:,1].max()])
        with torch.inference_mode():
            masks, _, _ = predictor.predict(box=box, multimask_output=False)
        m = masks[0].astype(np.uint8)
        gt = np.zeros((H, W), np.uint8); cv2.fillPoly(gt, [pk.astype(np.int32)], 1)
        inter = int((m & gt).sum()); union = int((m | gt).sum())
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        a_rect = 0.0
        if cnts:
            (cw, ch) = cv2.minAreaRect(max(cnts, key=cv2.contourArea))[1]; a_rect = cw*ch
        records.append(dict(iou=inter/union if union else 0.0,
                            a_mask=float(m.sum()), a_gt=float(gt.sum()), a_rect=a_rect))
        if len(examples) < 4:
            examples.append((im, pk, m, box))

import pandas as pd
d = pd.DataFrame(records)
pe_mask = (d.a_mask - d.a_gt).abs()/d.a_gt*100
pe_rect = (d.a_rect - d.a_gt).abs()/d.a_gt*100
print(f"SAM2 area validation on {len(d)} GT arrays (GT-box prompted -> mask ceiling):")
print(f"  median mask<->GT IoU       : {d.iou.median():.3f}")
print(f"  median |area err| (raw mask): {pe_mask.median():.1f}%")
print(f"  median |area err| (minRect) : {pe_rect.median():.1f}%   <- the production recipe")
print(f"  @ GSD={GSD_M} m/px  median GT array = {d.a_gt.median()*GSD_M**2:.1f} m2")

In [ ]:
# --- SAM overlays: GT polygon (green) vs SAM mask (red) ------------------
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(examples), figsize=(4*len(examples), 4))
if len(examples) == 1:
    axes = [axes]
for ax, (im, pk, m, box) in zip(axes, examples):
    x1, y1, x2, y2 = [int(v) for v in box]; pad = 20
    xa, ya = max(0, x1-pad), max(0, y1-pad); xb, yb = x2+pad, y2+pad
    crop = im[ya:yb, xa:xb].copy()
    ov = crop.copy(); ov[m[ya:yb, xa:xb] > 0] = [255, 0, 0]
    crop = (0.5*crop + 0.5*ov).astype(np.uint8)
    cv2.polylines(crop, [(pk - [xa, ya]).astype(np.int32)], True, (0, 255, 0), 2)
    ax.imshow(crop); ax.axis('off'); ax.set_title('GT=green  SAM=red')
plt.tight_layout(); plt.show()